# NumPy Fundamentals & ndarray Architecture Part-I

> This notebook contains an exhaustive suite of production-grade interview flashcards covering core NumPy mechanics, multidimensional array operations, indexing, and runtime behavior.

### Topics Covered:
* NumPy contiguous memory model vs. Python list pointer overhead
* 1D, 2D, and 3D array creation and axis semantics
* Implicit type upcasting, Unicode strings, and explicit `dtype` control
* Inspection attributes: `ndim`, `shape`, `size`, `dtype`, `itemsize`, and `data`
* Positional integer indexing, multidimensional coordinate indexing, and boolean masking
* Basic slicing, multidimensional slicing, and the Ellipsis (`...`) operator
* Vectorized element-wise arithmetic and broadcasting compatibility constraints



### 1. What is NumPy and why is it preferred over native Python lists for scientific computing?

**Technical Breakdown**: NumPy (Numerical Python) is built around the `ndarray` object, which encapsulates a contiguous block of C-style homogeneous memory. Native Python lists are arrays of pointers pointing to fragmented `PyObject` structures scattered across the heap, incurring significant pointer-chasing and dynamic type-checking overhead. NumPy executes vector operations at C-speed with cache locality and minimal memory overhead.

In [1]:
import numpy as np

py_list = [1, 2, 3]
np_arr = np.array([1, 2, 3])

print("Python List:", py_list)
print("NumPy Array:", np_arr)

Python List: [1, 2, 3]
NumPy Array: [1 2 3]


### 2. How is NumPy installed across different package management ecosystems?

**Technical Breakdown**: In standard environments, NumPy is installed via PyPI wheels using `pip install numpy`. In scientific distributions, it is managed via conda using `conda install numpy`, which links precompiled binaries against platform-optimized linear algebra libraries like Intel MKL or OpenBLAS.

---
- BLAS — Basic Linear Algebra Subprograms: Rules for performing mathematical calculations efficiently.
- Intel MKL — Intel Math Kernel Library: An Intel-developed library that speeds up mathematical calculations.
- OpenBLAS — Open-source Basic Linear Algebra Subprograms: A free, open-source library that speeds up mathematical calculations.

In short:
* Intel MKL and OpenBLAS are two different libraries that help computers perform maths faster.
* Both libraries produce the same correct mathematical results.
* They use different computer instructions and optimization techniques to perform the calculations. Their speed depends on your processor and the type of maths.

NumPy can use either library behind the scenes, so you can focus on writing your code!

In [2]:
import numpy as np

print("NumPy Version:", np.__version__)
config = np.show_config(mode="dicts")
print("blas:")
print(f"    name: {config['Build Dependencies']['blas']['name']}")

NumPy Version: 2.1.3
blas:
    name: scipy-openblas


### 3. What is the standard convention for importing NumPy and why is namespace aliasing enforced?

**Technical Breakdown**: The community convention is `import numpy as np`. This maintains clean namespace separation, avoiding clashes with Python's built-in functions (e.g., `sum`, `min`, `max`, `abs`) while reducing syntax verbosity during intensive numerical programming.

---
- **Readability:** np.array() is concise and widely recognized.

- **Consistency:** Python developers generally use np for NumPy.

- **Avoids conflicts:** The np prefix distinguishes NumPy functions from similarly named functions in other libraries.

In [3]:
import numpy as np
print("NumPy Namespace Alias Verified:", np is not None)

NumPy Namespace Alias Verified: True


### 4. What are the key architectural characteristics of an array compared to generic collections?

**Technical Breakdown**: An array is an ordered data structure containing elements of strictly identical data types stored in contiguous physical memory addresses. This guarantees $O(1)$ random access time via base-address offset arithmetic: $\text{Address} = \text{Base} + (\text{Index} \times \text{Itemsize})$.

In [4]:
import numpy as np

arr = np.array([10, 9, 99, 71, 90])
print("Array Elements:", arr)
base = arr.ctypes.data
itemsize = arr.itemsize
print("Base Address:", base)
print("Itemsize:", itemsize)

for index, value in enumerate(arr):
    address = base + (index * itemsize)
    print(f"Index: {index}, Value: {value}, Address: {address}")

Array Elements: [10  9 99 71 90]
Base Address: 618005376
Itemsize: 8
Index: 0, Value: 10, Address: 618005376
Index: 1, Value: 9, Address: 618005384
Index: 2, Value: 99, Address: 618005392
Index: 3, Value: 71, Address: 618005400
Index: 4, Value: 90, Address: 618005408


### 5. How do Python lists and NumPy `ndarray` objects compare in mutability and type flexibility?

**Technical Breakdown**: Both Python lists and NumPy arrays are mutable in-place. However, Python lists are heterogeneous, holding references to arbitrary types at the cost of high memory overhead. NumPy arrays enforce homogeneity, storing unboxed raw primitives directly in memory buffers.


In [5]:
import numpy as np

arr = np.array([1, 2, 3])
arr[0] = 99
print("Mutated Array:", arr)

Mutated Array: [99  2  3]


### 6. How is a 1D `ndarray` instantiated from a Python list?

**Technical Breakdown**: The function `np.array(object, ...)` accepts any sequence-like object. When passing a flat list of integers, NumPy inspects the sequence length, allocates a contiguous memory buffer of size $N \times \text{sizeof(int)}$, and copies the values.

In [6]:
import numpy as np

array_from_list = np.array([1, 2, 3, 4])
print("1-D array from list:", array_from_list)
print("Array Type:", type(array_from_list))
print("Array Dimensions:", array_from_list.ndim)
print("Array Shape:", array_from_list.shape)


1-D array from list: [1 2 3 4]
Array Type: <class 'numpy.ndarray'>
Array Dimensions: 1
Array Shape: (4,)


### 7. How is a 1D `ndarray` instantiated from a Python tuple?

**Technical Breakdown**: Instantiating an array from an immutable tuple operates identically to list conversion. The elements are unpacked and converted into a mutable C-contiguous buffer.

In [7]:
import numpy as np

array_from_tuple = np.array((1, 2, 3, 4))
print("1-D array from tuple:", array_from_tuple)
print("Is mutable:", array_from_tuple.flags.writeable)

1-D array from tuple: [1 2 3 4]
Is mutable: True


### 8. What occurs when passing multiple positional arguments instead of a sequence to `np.array()`?

**Technical Breakdown**: The signature of `np.array(object, dtype=None, ...)` accepts the sequence as its first positional parameter. Passing bare comma-separated values causes Python to interpret them as positional arguments 2, 3, and 4, raising a `TypeError`.

In [8]:
import numpy as np

try:
    a = np.array(1, 2, 3, 4)
    print("Array:", a)
except TypeError as e:
    print("Caught expected exception:", type(e).__name__, "-", e)

Caught expected exception: TypeError - array() takes from 1 to 2 positional arguments but 4 were given


### 9. How do you verify the internal class identity of a NumPy array object?

**Technical Breakdown**: Calling the built-in `type()` on an array returns `<class 'numpy.ndarray'>`. Internally, it is an instance of the C-extension type `PyArray_Type`.

> **Remember:** `ndarray` is short for N-dimensional array. The N means it can have one, two, three, or more dimensions.

In [9]:
import numpy as np

arr = np.array([1, 2, 3])
print("Type of array:", type(arr))
print("Is instance of np.ndarray:", isinstance(arr, np.ndarray))

Type of array: <class 'numpy.ndarray'>
Is instance of np.ndarray: True


### 10. How is a 2D `ndarray` constructed from nested Python lists?

**Technical Breakdown**: A 2D array (matrix) is created by passing a sequence of equally sized sub-sequences. NumPy creates a 2D stride descriptor over a single continuous memory block rather than creating multiple separate pointer arrays.

In [10]:
import numpy as np

array_2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("2-D array (matrix):\n", array_2d)

2-D array (matrix):
 [[1 2 3]
 [4 5 6]
 [7 8 9]]


### 11. How are axes defined and traversed in a 2D NumPy array?

**Technical Breakdown**: In a 2D array, `Axis 0` refers to the outer dimension (rows), running vertically downwards. `Axis 1` refers to the inner dimension (columns), running horizontally across columns.

> Multiple 1D arrays of same size

In [11]:
import numpy as np

arr_2d = np.array([[1, 2, 3], [4, 5, 6]])
print("2-D array (matrix):\n", arr_2d)
print("Shape of array (rows, cols):", arr_2d.shape)
print("Axis 0 reduction (sum over row axis):", np.sum(arr_2d, axis=0)) # (1,4) (2,5) (3,6)
print("Axis 1 reduction (sum over column axis):", np.sum(arr_2d, axis=1)) # (1, 2, 3) (4, 5, 6)

2-D array (matrix):
 [[1 2 3]
 [4 5 6]]
Shape of array (rows, cols): (2, 3)
Axis 0 reduction (sum over row axis): [5 7 9]
Axis 1 reduction (sum over column axis): [ 6 15]


### 12. How is a 3D `ndarray` constructed and mentally structured?

**Technical Breakdown**: A 3D array represents nested matrices stacked along a depth axis. It is passed as a list of lists of lists, with dimensions ordered as `(depth/matrices, rows, columns)`.

> Multiple 2D arrays of same shape

In [12]:
import numpy as np

array_3d = np.array([
    [
        ["B0R0C0", "B0R0C1", "B0R0C2", "B0R0C3"],
        ["B0R1C0", "B0R1C1", "B0R1C2", "B0R1C3"],
        ["B0R2C0", "B0R2C1", "B0R2C2", "B0R2C3"]
    ],
    [
        ["B1R0C0", "B1R0C1", "B1R0C2", "B1R0C3"],
        ["B1R1C0", "B1R1C1", "B1R1C2", "B1R1C3"],
        ["B1R2C0", "B1R2C1", "B1R2C2", "B1R2C3"]
    ]
])
print("B = Block, R = Row, C = Column")
print("Shape(Block, Row, Column):", array_3d.shape)
for i in range(3):
  print(list(range(array_3d.shape[i])),end=" ")
print("\n3-D array:\n", array_3d)



B = Block, R = Row, C = Column
Shape(Block, Row, Column): (2, 3, 4)
[0, 1] [0, 1, 2] [0, 1, 2, 3] 
3-D array:
 [[['B0R0C0' 'B0R0C1' 'B0R0C2' 'B0R0C3']
  ['B0R1C0' 'B0R1C1' 'B0R1C2' 'B0R1C3']
  ['B0R2C0' 'B0R2C1' 'B0R2C2' 'B0R2C3']]

 [['B1R0C0' 'B1R0C1' 'B1R0C2' 'B1R0C3']
  ['B1R1C0' 'B1R1C1' 'B1R1C2' 'B1R1C3']
  ['B1R2C0' 'B1R2C1' 'B1R2C2' 'B1R2C3']]]


### 13. What is the semantic role of each axis in a 3D NumPy array?

**Technical Breakdown**: For a 3D array, `Axis 0` represents the primary slices/matrices. `Axis 1` represents the rows within each matrix. `Axis 2` represents the individual columns within each row.

In [13]:
import numpy as np

arr_3d = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print("Axis 0 length (matrices):", arr_3d.shape[0])
print("Axis 1 length (rows):", arr_3d.shape[1])
print("Axis 2 length (columns):", arr_3d.shape[2])


Axis 0 length (matrices): 2
Axis 1 length (rows): 2
Axis 2 length (columns): 2


### 14. How does explicit type casting operate via the `dtype` argument during array initialization?

**Technical Breakdown**: Passing `dtype=float` (or `np.float64`) forces NumPy to allocate the buffer using 64-bit IEEE-754 floating-point slots, automatically coercing incoming integer values to floats.

In [14]:
import numpy as np

nested_list = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
array_float = np.array(nested_list, dtype=float)
print("Array with float dtype:\n", array_float)
print("Array dtype:", array_float.dtype)

Array with float dtype:
 [[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]
Array dtype: float64


### 15. What is implicit type promotion when mixing integers and floats?

**Technical Breakdown**: NumPy adheres to strict type promotion rules. When numerical types are mixed, NumPy upcasts all elements to the lowest common denominator type that preserves precision without overflow (integers upcast to `float64`).


In [15]:
import numpy as np

array3 = np.array([[2.4, 3], [4.91, 7], [0, -1]])
print("Upcasted Array:\n", array3)
print("Resulting dtype:", array3.dtype)

Upcasted Array:
 [[ 2.4   3.  ]
 [ 4.91  7.  ]
 [ 0.   -1.  ]]
Resulting dtype: float64


### 16. How does NumPy handle heterogeneous sequences containing strings and numbers?
**Technical Breakdown**: If any string is present in a mixed sequence without an explicit object dtype, NumPy upcasts all numeric elements to Unicode string format (`<U32` or similar), maintaining buffer homogeneity rather than failing.

In [16]:
import numpy as np

array2 = np.array([5, -7.4, 'a', 7.2])
print("Auto-promoted Unicode array:", array2)
print("Inferred dtype:", array2.dtype)

Auto-promoted Unicode array: ['5' '-7.4' 'a' '7.2']
Inferred dtype: <U32


C-vectorization means arranging array elements in memory in the order used by the C programming language. In NumPy, this is called C-order (row-major order).

For your array of shape `(2, 3, 4)`:

```
import numpy as np
arr = np.arange(24).reshape(2, 3, 4)
print(arr.ravel(order='C'))
```

Output:

```
[ 0  1  2  3  4  5  6  7  8  9 10 11
 12 13 14 15 16 17 18 19 20 21 22 23]
```

The elements are laid out in this order:

```
B0:
R0:  0   1   2   3
R1:  4   5   6   7
R2:  8   9  10  11

B1:
R0: 12  13  14  15
R1: 16  17  18  19
R2: 20  21  22  23
```

How to remember it:

- First, move across the columns.
- Then move down the rows.
- Finally, move to the next block.

For a standard contiguous NumPy array, the last axis (columns) changes fastest in C-order.

Note: C-order is about memory layout and element ordering, not vectorization in the sense of executing operations on multiple data elements at once.


### 17. How can you store mixed data types without automatic string upcasting?
**Technical Breakdown**: Passing `dtype=object` forces NumPy to store pointers to arbitrary Python objects rather than unboxed primitives. This bypasses homogeneity but loses C-vectorization efficiency and cache locality.



In [17]:
import numpy as np

arr_obj = np.array([5, -7.4, 'a', 7.2], dtype=object)
print("Object Array:", arr_obj)
print("Array dtype:", arr_obj.dtype)
print("Element types:", [type(x) for x in arr_obj])

Object Array: [5 -7.4 'a' 7.2]
Array dtype: object
Element types: [<class 'int'>, <class 'float'>, <class 'str'>, <class 'float'>]


### 18. What information does the `ndarray.ndim` attribute expose?
**Technical Breakdown**: The `ndim` attribute returns an integer denoting the total number of axes/dimensions (rank) of the array. It equals `len(arr.shape)`.

In [18]:
import numpy as np

arr1d = np.array([1, 2, 3])
arr2d = np.array([[1, 2, 3], [4, 5, 6]])
arr3d = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

print(f"arr1d ndim: {arr1d.ndim} = {len(arr1d.shape)} len(arr1d.shape({arr1d.shape}))")
print(f"arr2d ndim: {arr2d.ndim} = {len(arr2d.shape)} len(arr2d.shape({arr2d.shape}))")
print(f"arr3d ndim: {arr3d.ndim} = {len(arr3d.shape)} len(arr3d.shape({arr3d.shape}))")



arr1d ndim: 1 = 1 len(arr1d.shape((3,)))
arr2d ndim: 2 = 2 len(arr2d.shape((2, 3)))
arr3d ndim: 3 = 3 len(arr3d.shape((2, 2, 2)))


### 19. What is the return type and format of `ndarray.shape` for a 1D array?
**Technical Breakdown**: For a 1D array of length N, `arr.shape` returns a single-element tuple `(N,)`. The trailing comma distinguishes it syntactically from an integer in parentheses.

In [19]:
import numpy as np

arr_1d = np.array([1, 2, 3, 4, 5])
print("Shape of 1D array:", arr_1d.shape)
print("Tuple length:", len(arr_1d.shape))
print("Is tuple:", isinstance(arr_1d.shape, tuple))

Shape of 1D array: (5,)
Tuple length: 1
Is tuple: True


### 20. How does `ndarray.shape` describe a 2D matrix layout?
**Technical Breakdown**: For a 2D array, `arr.shape` returns a 2-tuple `(d0, d1)` where `d0` represents the number of rows and `d1` represents the number of columns.


In [20]:
import numpy as np

arr_2d = np.array([[1, 2, 3], [4, 5, 6]])
print("Shape of 2D array:", arr_2d.shape)
print("Rows:", arr_2d.shape[0], "Columns:", arr_2d.shape[1])

Shape of 2D array: (2, 3)
Rows: 2 Columns: 3


### 21. How does `ndarray.shape` scale to 3D and higher-dimensional structures?
**Technical Breakdown**: For a 3D array, `shape` returns `(d0, d1, d2)` where `d0` denotes the number of slices/matrices, `d1` the rows per matrix, and `d2` the columns per row.

In [21]:
import numpy as np

arr_3d = np.array([
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]],
    [[9, 10], [11, 12]]
])
print("Shape of 3D array:", arr_3d.shape)

Shape of 3D array: (3, 2, 2)


### 22. What does the `ndarray.size` attribute calculate and how does it correlate with `shape`?
**Technical Breakdown**: The `size` property returns the total count of elements across all dimensions. Mathematically, it equals the product of the shape tuple elements: size = prod(shape[i]).


In [22]:
import numpy as np

arr_2d = np.array([[1, 2, 3], [4, 5, 6]])
arr_3d = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]], [[9, 10], [11, 12]]])
#np.prod(arr_3d.shape) == (arr_3d.shape[0] * arr_3d.shape[1] * arr_3d.shape[2])
print("arr_2d size:", arr_2d.size)
print("Matches product:", arr_2d.size == (arr_2d.shape[0] * arr_2d.shape[1]))
print("arr_3d size:", arr_3d.size)
print("Matches product:", arr_3d.size == (arr_3d.shape[0] * arr_3d.shape[1] * arr_3d.shape[2]))

arr_2d size: 6
Matches product: True
arr_3d size: 12
Matches product: True


### 23. How does `ndarray.dtype` reflect primitive storage formats?
**Technical Breakdown**: The `dtype` attribute describes the memory layout, bit-width, and byte-order of each element. Common defaults include `int32`/`int64` for integers and `float64` for standard floating-point numbers.

In [23]:
import numpy as np

arr_int = np.array([1, 2, 3])
arr_float = np.array([1.0, 2.5, 3.7])

print("Integer dtype:", arr_int.dtype)
print("Float dtype:", arr_float.dtype)

Integer dtype: int64
Float dtype: float64


### 24. How does `ndarray.dtype` encode fixed-width Unicode strings?
**Technical Breakdown**: NumPy assigns `<U[N]` to string arrays, where `<` indicates little-endian byte ordering, `U` denotes UTF-32 Unicode, and `N` is the maximum character length found among the elements.


In [24]:
import numpy as np

arr_str = np.array(['apple', 'banana', 'cherry'])
print("String array dtype:", arr_str.dtype)

String array dtype: <U6


### 25. What does `ndarray.itemsize` represent and how is it related to `dtype`?
**Technical Breakdown**: The `itemsize` property returns the memory footprint of an individual element in bytes. An `int32` array yields 4 bytes, `float64` yields 8 bytes, and `<U6` (UTF-32 string of 6 chars) yields 24 bytes (6 * 4 bytes per UTF-32 char).
```
32 bits = 4 bytes
64 bits = 8 bytes
```

In [25]:
import numpy as np

arr_int = np.array([1, 2, 3, 4, 5], dtype='int32')
arr_float = np.array([1.0, 2.5, 3.7], dtype='float64')
arr_str = np.array(['apple', 'banana', 'cherry'])

print("Itemsize arr_int:", arr_int.itemsize, "bytes")
print("Itemsize arr_float:", arr_float.itemsize, "bytes")
print("Itemsize arr_str:", arr_str.itemsize, "bytes")

Itemsize arr_int: 4 bytes
Itemsize arr_float: 8 bytes
Itemsize arr_str: 24 bytes


### 26. How do you calculate the total byte footprint of an `ndarray`?
**Technical Breakdown**: The total volume of memory consumed by the array data buffer is given by `arr.size * arr.itemsize` (which matches the `arr.nbytes` attribute).

In [26]:
import numpy as np

arr = np.array([[1, 2, 3], [4, 5, 6]], dtype='float64')
total_bytes = arr.size * arr.itemsize
print("Calculated total bytes:", total_bytes)
print("Matches arr.nbytes:", total_bytes == arr.nbytes)

Calculated total bytes: 48
Matches arr.nbytes: True


### 27. What does the `ndarray.data` attribute expose?
**Technical Breakdown**: The `arr.data` attribute returns a Python `<memoryview>` referencing the array's actual memory buffer in RAM. It allows low-level buffer-protocol operations without intermediate copying.


In [27]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5])
data_buffer = arr.data
print("Data buffer:", data_buffer)
print("Type of data buffer:", type(data_buffer))

Data buffer: <memory at 0x7b229ef34940>
Type of data buffer: <class 'memoryview'>


### 28. How does 1D integer indexing retrieve single elements?
**Technical Breakdown**: 1D indexing uses zero-based integer offsets to look up elements.


In [28]:
import numpy as np

arr_1d = np.array([1, 2, 3, 4, 5])
val = arr_1d[2]
print("Element at index 2:", val)

Element at index 2: 3


### 29. How does 2D integer indexing differ between tuple syntax and chained index syntax?
**Technical Breakdown**: Multi-dimensional arrays are indexed using comma-separated coordinates `arr[row, col]`. While chained indexing `arr[row][col]` works, it is inefficient because `arr[row]` materializes an intermediate 1D array view before applying the second lookup.
> Direct indexing - Access row 1, column 2 directly
```python
arr_2d[1, 2]
# Row 1, Column 2 → 60
```
> Chained indexing - First get row 1, then element 2
```python
arr_2d[1]     # [40, 50, 60]
arr_2d[1][2]  # 60
```


In [29]:
import timeit
import numpy as np

arr_2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])

print("Direct tuple indexing:", arr_2d[1, 2]) # One indexing operation
print("Chained indexing:", arr_2d[1][2]) # Two indexing operation

# Measure execution time over 1 million repetitions
direct_time = timeit.timeit(lambda: arr_2d[1, 2], number=1_000_000)
chained_time = timeit.timeit(lambda: arr_2d[1][2], number=1_000_000)

print(f"\nDirect indexing: {direct_time:.6f} seconds")
print(f"Chained indexing: {chained_time:.6f} seconds")
print(f"Direct indexing per call: {direct_time / 1_000_000 * 1e9:.2f} ns")
print(f"Chained indexing per call: {chained_time / 1_000_000 * 1e9:.2f} ns")

Direct tuple indexing: 6
Chained indexing: 6

Direct indexing: 0.311148 seconds
Chained indexing: 0.323684 seconds
Direct indexing per call: 311.15 ns
Chained indexing per call: 323.68 ns


### 30. How is multi-dimensional coordinate indexing implemented in a 3D array?
**Technical Breakdown**: For a 3D array, coordinate access takes three indices: `arr[matrix_idx, row_idx, col_idx]`. NumPy uses internal stride values to jump across the dimensional planes directly to the targeted byte offset.

In [30]:
import numpy as np

arr_3d = np.array([
    [[1, 2, 3], [4, 5, 6]],
    [[7, 8, 9], [10, 11, 12]],
    [[13, 14, 15], [16, 17, 18]]
])

print("Element at [1, 0, 2]:", arr_3d[1, 0, 2])

Element at [1, 0, 2]: 9


### 31. How does 1D boolean masking filter array elements?
**Technical Breakdown**: Passing a boolean array of identical length evaluates each element's condition. Elements corresponding to `True` are retained; elements corresponding to `False` are omitted.


In [31]:
import numpy as np

# 1D array representing people's ages
arr_1d = np.array([10, 20, 30, 40, 50])

# Filter using a Boolean mask
mask_1d = np.array([True, False, True, False, True])
filtered = arr_1d[mask_1d]
print("Ages selected by Boolean mask:", filtered)

# Filter people older than 25
filtered = arr_1d[arr_1d > 25]
print("Ages above 25:", filtered)

Ages selected by Boolean mask: [10 30 50]
Ages above 25: [30 40 50]


### 32. Why does boolean masking on a 2D array return a flattened 1D array?
**Technical Breakdown**: A boolean mask can select an irregular number of elements per row. Because an `ndarray` must have fixed rectangular dimensions, non-uniform selections cannot form a 2D matrix; NumPy collapses and flattens the output into a 1D array.

In [32]:
import numpy as np

arr_2d = np.array([[10, 20, 30], [40, 50, 60], [70, 80, 90]])
mask_2d = np.array([
    [True, False, True],
    [False, True, False],
    [True, False, True]
])
result = arr_2d[mask_2d]
print("Filtered 2D Result (Flattened):\n", result)
print("Result Shape:", result.shape)

Filtered 2D Result (Flattened):
 [10 30 50 70 90]
Result Shape: (5,)


### 33. How does boolean masking behave across 3D arrays?
**Technical Breakdown**: Similar to 2D arrays, applying a 3D boolean mask extracts all values where the mask evaluates to `True`, flattening the selected items into a contiguous 1D sequence.

In [33]:
import numpy as np

arr_3d = np.array([
    [[10, 20], [30, 40]],
    [[50, 60], [70, 80]],
    [[90, 100], [110, 120]]
])
mask_3d = np.array([
    [[True, False], [False, True]],
    [[True, False], [False, True]],
    [[True, False], [False, True]]
])
result_3d = arr_3d[mask_3d]
print("3D array with mask:\n", result_3d)
print("Result ndim:", result_3d.ndim)

3D array with mask:
 [ 10  40  50  80  90 120]
Result ndim: 1


### 34. What are the boundary semantics of basic slicing `start:stop:step`?
**Technical Breakdown**: Slicing operates with inclusive `start` and exclusive `stop` indices. Unlike scalar indexing which returns an element, basic slicing constructs a view of the array sharing the underlying memory buffer.

In [34]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
slice_res = arr[2:6]
print("Sliced arr[2:6]:", slice_res)

Sliced arr[2:6]: [3 4 5 6]


### 35. How does strided slicing operate using the step argument?
**Technical Breakdown**: Using syntax `arr[::step]` modifies the stride step size across the array buffer. For instance, `step=2` skips every other element without copying data.

In [35]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9])
strided_res = arr[::2]
print("Strided arr[::2]:", strided_res)
arr_reversed = arr[::-1]
print("Reversed arr[::-1]:", arr_reversed)

Strided arr[::2]: [1 3 5 7 9]
Reversed arr[::-1]: [9 8 7 6 5 4 3 2 1]


### 36. How is multidimensional submatrix slicing performed on a 2D array?
**Technical Breakdown**: Slicing syntax is extended across axes using commas: `arr[r_start:r_stop, c_start:c_stop]`. Rows and columns are sliced simultaneously along Axis 0 and Axis 1.

In [36]:
import numpy as np

arr_2d = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])
subarray = arr_2d[1:3, 1:3]
print("Subarray from 2D array:\n", subarray)

Subarray from 2D array:
 [[ 6  7]
 [10 11]]


### 37. How can you slice a subset of rows across all columns in a 2D array?
**Technical Breakdown**: Specifying `arr[:k]` slices the first k rows across Axis 0. Any trailing axes that are omitted default to full slices (`:`).

In [37]:
import numpy as np

array_2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Slice first two rows:\n", array_2d[:2])

Slice first two rows:
 [[1 2 3]
 [4 5 6]]


### 38. How can you slice a subset of columns across all rows in a 2D array?
**Technical Breakdown**: Using an explicit colon for the first dimension `arr[:, start:]` preserves all rows while slicing the columns starting from `start` to the end.

In [38]:
import numpy as np

array_2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Slice last two columns:\n", array_2d[:, 1:])

Slice last two columns:
 [[2 3]
 [5 6]
 [8 9]]


### 39. How do mixed scalar indexing and slicing interact in 3D arrays?
**Technical Breakdown**: Supplying a slice for some dimensions and a scalar for others reduces the dimensionality of the result by 1 for each scalar index used. Slicing `[0:2, 1, 0:2]` extracts a 2D array of shape `(2, 2)` from a 3D array.

In [39]:
import numpy as np

arr_3d = np.array([
    [[1, 2, 3], [4, 5, 6]],
    [[7, 8, 9], [10, 11, 12]],
    [[13, 14, 15], [16, 17, 18]]
])
subarray_3d = arr_3d[0:2, 1, 0:2]
print("Subarray from 3D array:\n", subarray_3d)
print("Subarray Shape:", subarray_3d.shape)

Subarray from 3D array:
 [[ 4  5]
 [10 11]]
Subarray Shape: (2, 2)


### 40. How does the Ellipsis (`...`) operator simplify multidimensional array slicing?
**Technical Breakdown**: The ellipsis literal `...` expands to as many `:` full slices as necessary to match the total number of unmentioned axes in an array. `arr[..., :]` selects all elements across preceding dimensions.

In [40]:
import numpy as np

arr_3d = np.array([
    [[1, 2, 3], [4, 5, 6]],
    [[7, 8, 9], [10, 11, 12]],
    [[13, 14, 15], [16, 17, 18]]
])
subarray = arr_3d[..., :]
print("Subarray with ellipsis shape:", subarray.shape)

Subarray with ellipsis shape: (3, 2, 3)


### 41. How is the Ellipsis used to slice specific rows across all matrices and columns in a 3D array?
**Technical Breakdown**: The syntax `arr[:, 0, ...]` selects all matrices along Axis 0, extracts row index 0 along Axis 1, and uses `...` to capture all remaining trailing dimensions (Axis 2 columns).

In [41]:
import numpy as np

arr_3d = np.array([
    [[1, 2, 3], [4, 5, 6]],
    [[7, 8, 9], [10, 11, 12]],
    [[13, 14, 15], [16, 17, 18]]
])
subarray = arr_3d[:, 0, ...]
print("Sliced via [:, 0, ...]:\n", subarray)
print("Resulting shape:", subarray.shape)

Sliced via [:, 0, ...]:
 [[ 1  2  3]
 [ 7  8  9]
 [13 14 15]]
Resulting shape: (3, 3)


### 42. How does element-wise arithmetic addition, subtraction, and multiplication operate?
**Technical Breakdown**: Binary operators (`+`, `-`, `*`) execute element-by-element over corresponding index pairs in O(N) time. The `*` operator computes the Hadamard (element-wise) product, not matrix multiplication.

In [42]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("Addition (a + b):", a + b)
print("Subtraction (b - a):", b - a)
print("Multiplication (a * b):", a * b)

Addition (a + b): [5 7 9]
Subtraction (b - a): [3 3 3]
Multiplication (a * b): [ 4 10 18]


### 43. How do division, modulus, exponentiation, and floor division operate on arrays?
**Technical Breakdown**: True division `/` performs float division element-wise. Modulus `%` computes remainders, `**` raises elements to the given power, and `//` performs integer truncation.

In [43]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

print("Division (b / a):", b / a)
print("Modulus (b % a):", b % a)
print("Exponentiation (a ** 2):", a ** 2)
print("Floor Division (b // a):", b // a)

Division (b / a): [4.  2.5 2. ]
Modulus (b % a): [0 1 0]
Exponentiation (a ** 2): [1 4 9]
Floor Division (b // a): [4 2 2]


### 44. What happens when element-wise arithmetic operations are attempted on arrays with mismatched shapes?
**Technical Breakdown**: When array dimensions cannot be aligned or broadcast, NumPy throws a `ValueError: operands could not be broadcast together`. Element-wise operations require compatible shapes across all corresponding dimensions.

In [44]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([1, 2, 3, 4])

try:
    c = a + b
    print("Result:", c)
except ValueError as e:
    print("Caught expected exception:", type(e).__name__, "-", e)

Caught expected exception: ValueError - operands could not be broadcast together with shapes (3,) (4,) 


### 45. What are the common statistical helper functions provided directly by NumPy?
**Technical Breakdown**: Beyond array creation and slicing, NumPy provides built-in statistical routines such as `np.mean()`, `np.median()`, `np.std()`, and `np.var()` to analyze data distributions directly on arrays.


In [45]:
import numpy as np

data = np.array([10, 50, 30, 20, 40])
print("Mean:", np.mean(data))
print("Median:", np.median(data))
print("Standard Deviation:", np.std(data))
print("Variance:", np.var(data))

Mean: 30.0
Median: 30.0
Standard Deviation: 14.142135623730951
Variance: 200.0
